# Digital Image Processing Lab Final — Colab Reference

Each Markdown cell explains the code immediately below it. Run the first three cells, then run any required topic cell during the exam.


## Cell 1 — Install and import packages

Run this first in Google Colab.


In [ ]:
!pip -q install seaborn-image

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import requests
from io import BytesIO
from PIL import Image, ImageOps

import skimage
import skimage as ski
from skimage import data, color, exposure, filters, segmentation
from skimage.util import random_noise
from skimage.exposure import match_histograms
from skimage.color import label2rgb
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.draw import circle_perimeter
from skimage.feature import canny
from skimage.morphology import erosion, dilation
from scipy.ndimage import convolve, median_filter, label
from scipy.fftpack import dct, idct
import seaborn_image as isns


## Cell 2 — Utility function to show image results

Use this function to display images in a labeled grid.


In [ ]:
def img_grid(images, title='Image Plot', subtitles=None, cols=3, figsize=(18, 10), cmap='gray'):
    if subtitles is None:
        subtitles = [''] * len(images)
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    fig.suptitle(title, fontsize=16, y=1.02)
    for ax, image, subtitle in zip(axes, images, subtitles):
        arr = np.array(image) if isinstance(image, Image.Image) else image
        ax.imshow(arr, cmap=cmap if arr.ndim == 2 else None)
        ax.set_title(subtitle)
        ax.axis('off')
    for ax in axes[len(images):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


## Cell 3 — Load the working image from a URL

This loads an RGB image, converts it to grayscale, prints its size, and displays it.


In [ ]:
url = 'https://fatcatart.com/wp-content/uploads/2019/03/Van-Gogh-Starry-Night-cat-w.jpg'
response = requests.get(url)
PIL_img = Image.open(BytesIO(response.content)).convert('RGB')
PIL_img_gray = PIL_img.convert('L')
gray_8bit = np.array(PIL_img_gray)
print('Image resolution:', PIL_img.size)
img_grid([PIL_img], 'Original Image', ['RGB Image'], cols=1, figsize=(8, 6))


## Cell 4 — Read a local image with OpenCV

Upload a file to Colab, read it with OpenCV, convert BGR to RGB, then display it.


In [ ]:
# from google.colab import files
# uploaded = files.upload()
# image_path = list(uploaded.keys())[0]
# image_bgr = cv2.imread(image_path)
# image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
# print('OpenCV image shape:', image_bgr.shape)
# plt.imshow(image_rgb)
# plt.axis('off')
# plt.title('BGR Converted to RGB')
# plt.show()


## Cell 5 — Show grayscale and negative images

This displays original RGB, grayscale, RGB negative, and grayscale negative images.


In [ ]:
negative_rgb = ImageOps.invert(PIL_img)
negative_gray = ImageOps.invert(PIL_img_gray)
img_grid([PIL_img, PIL_img_gray, negative_rgb, negative_gray],
         'Grayscale and Negatives',
         ['Original RGB', 'Grayscale', 'RGB Negative', 'Grayscale Negative'], cols=4)


## Cell 6 — Extract red, green, and blue channels

This splits an RGB image into red, green, and blue channel images.


In [ ]:
r, g, b = PIL_img.split()
img_grid([PIL_img, r, g, b], 'RGB Channels', ['Original', 'Red', 'Green', 'Blue'], cols=4)


## Cell 7 — Display individual channels in their actual colors

This creates images where only the red, green, or blue component is visible.


In [ ]:
def colored_channels(image):
    channels = image.split()
    shape = np.array(image).shape
    outputs = []
    for i in range(3):
        output = np.zeros(shape, dtype=np.uint8)
        output[:, :, i] = np.array(channels[i])
        outputs.append(Image.fromarray(output, 'RGB'))
    return outputs

r_color, g_color, b_color = colored_channels(PIL_img)
img_grid([PIL_img, r_color, g_color, b_color],
         'Colored RGB Channels', ['Original', 'Red Only', 'Green Only', 'Blue Only'], cols=4)


## Cell 8 — Reduce spatial resolution

This resizes an image to 128×128 using bicubic interpolation.


In [ ]:
low_res = PIL_img.resize((128, 128), Image.BICUBIC)
print('Low-resolution size:', low_res.size)
img_grid([PIL_img, low_res], 'Spatial Resolution', ['Original', '128 x 128'], cols=2)


## Cell 9 — Reduce intensity resolution from 8-bit to 3-bit

A 3-bit image has 8 levels. Shift/reduce by 5 bits and rescale for visualization.


In [ ]:
bit_reduction = 5
reduced = gray_8bit // (2 ** bit_reduction)
rescaled_3bit = (reduced * (255 // 7)).astype(np.uint8)
print('Reduced max:', reduced.max())
print('Reduced min:', reduced.min())
img_grid([gray_8bit, rescaled_3bit], 'Intensity Resolution', ['Original 8-bit', 'Reduced 3-bit'], cols=2)


## Cell 10 — Histogram and heatmap

This shows brightness distribution and actual intensity values in a small image grid.


In [ ]:
small_gray = cv2.resize(gray_8bit, (15, 15))
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(gray_8bit.flatten(), bins=30, kde=True, color='blue', ax=axes[0])
axes[0].set_title('Image Histogram')
axes[0].set_xlabel('Pixel Intensity')
sns.heatmap(small_gray, annot=True, fmt='d', cmap='gray', cbar=False, ax=axes[1])
axes[1].set_title('15 x 15 Heatmap')
plt.tight_layout(); plt.show()


# Lab 3 — Image Operations, Bit Planes, Masking, and Intensity Transformations


## Cell 11 — Brightness, rotation, and subtraction

This rotates an image, changes brightness with clipping, and subtracts two images.


In [ ]:
coins = ski.data.coins()
coins_rotated = (ski.transform.rotate(coins, 2) * 255).astype(np.uint8)
coins_bright = np.clip(coins_rotated.astype(np.int16) + 50, 0, 255).astype(np.uint8)
difference = np.clip(coins.astype(np.int16) - coins_rotated.astype(np.int16), 0, 255).astype(np.uint8)
img_grid([coins, coins_rotated, coins_bright, difference],
         'Basic Image Operations', ['Original', 'Rotated', 'Brightness +50', 'Original - Rotated'], cols=4)


## Cell 12 — Bit-plane slicing

This extracts all eight binary bit planes from a grayscale image.


In [ ]:
def bitplane_slice(image):
    planes = []
    for i in range(8):
        plane = (image & (1 << i))
        planes.append((plane > 0).astype(np.uint8) * 255)
    return planes

bitplanes = bitplane_slice(gray_8bit)
img_grid([gray_8bit] + bitplanes, 'Bit-Plane Slicing',
         ['Original'] + [f'Bitplane {i}' for i in range(8)], cols=3)


## Cell 13 — Reconstruct from selected bit planes

This reconstructs an approximation of the image using high-order bit planes.


In [ ]:
def reconstruct_from_bitplanes(planes, indices):
    output = np.zeros_like(planes[0], dtype=np.int64)
    for i in indices:
        output += (planes[i] // 255) * (2 ** i)
    return np.clip(output, 0, 255).astype(np.uint8)

high4 = reconstruct_from_bitplanes(bitplanes, [7, 6, 5, 4])
high2 = reconstruct_from_bitplanes(bitplanes, [7, 6])
img_grid([gray_8bit, high4, high2], 'Bit-Plane Reconstruction',
         ['Original', 'Planes 7,6,5,4', 'Planes 7,6'], cols=3)


## Cell 14 — Rectangular masking

This keeps only the right half of the RGB image.


In [ ]:
rgb_np = np.array(PIL_img)
h, w, c = rgb_np.shape
right_mask = np.zeros((h, w, 3), dtype=np.uint8)
right_mask[:, w // 2:, :] = 1
right_output = rgb_np * right_mask
img_grid([rgb_np, right_mask * 255, right_output],
         'Rectangular Masking', ['Original', 'Right-Half Mask', 'Output'], cols=3)


## Cell 15 — Circular masking

This keeps the area outside a circle. Change `>=` to `<=` to keep the inside.


In [ ]:
circle_mask = np.zeros((h, w, 3), dtype=np.uint8)
cx, cy = w // 2, h // 2
radius = 100
Y, X = np.ogrid[:h, :w]
outside_circle = (X - cx) ** 2 + (Y - cy) ** 2 >= radius ** 2
circle_mask[outside_circle] = 1
circle_output = rgb_np * circle_mask
img_grid([rgb_np, circle_mask * 255, circle_output],
         'Circular Masking', ['Original', 'Outside-Circle Mask', 'Output'], cols=3)


## Cell 16 — Log transformation

The log transform enhances dark areas.


In [ ]:
def log_transform(image, c=25):
    output = c * np.log(1 + image.astype(np.float32))
    return np.clip(output, 0, 255).astype(np.uint8)

log_image = log_transform(np.array(PIL_img))
img_grid([PIL_img, log_image], 'Log Transformation', ['Original', 'Log Transformed'], cols=2)


## Cell 17 — Gamma transformation

Gamma below 1 brightens the image; gamma above 1 darkens it.


In [ ]:
def gamma_transform(image, gamma, c=1):
    normalized = image.astype(np.float32) / 255.0
    output = c * normalized ** gamma
    return np.clip(255 * output, 0, 255).astype(np.uint8)

gamma_images = [np.array(PIL_img), gamma_transform(np.array(PIL_img), 0.5),
                gamma_transform(np.array(PIL_img), 0.2), gamma_transform(np.array(PIL_img), 1.5),
                gamma_transform(np.array(PIL_img), 2.5), gamma_transform(np.array(PIL_img), 10)]
img_grid(gamma_images, 'Gamma Transformation',
         ['Original', 'Gamma 0.5', 'Gamma 0.2', 'Gamma 1.5', 'Gamma 2.5', 'Gamma 10'], cols=3)


## Cell 18 — Histogram stretching

This expands the intensity range to 0–255.


In [ ]:
def histogram_stretching(image, min_val=0, max_val=255):
    old_min, old_max = image.min(), image.max()
    stretched = ((image - old_min) * ((max_val - min_val) / (old_max - old_min))) + min_val
    return np.clip(stretched, min_val, max_val).astype(np.uint8)

log_gray = log_transform(gray_8bit)
subtracted = np.clip(log_gray.astype(np.int16) - 100, 0, 255).astype(np.uint8)
stretched = histogram_stretching(subtracted)
img_grid([subtracted, stretched], 'Histogram Stretching', ['Input', 'Stretched'], cols=2)


# Lab 4 — Thresholding and Histogram Operations


## Cell 19 — Single thresholding at k = 128

This creates a binary mask and applies it to the grayscale image.


In [ ]:
k = 128
binary_mask = gray_8bit < k
binary_display = binary_mask.astype(np.uint8) * 255
threshold_output = gray_8bit * binary_mask.astype(np.uint8)
img_grid([gray_8bit, binary_display, threshold_output],
         'Single Thresholding', ['Original', 'Mask', 'Original x Mask'], cols=3)


## Cell 20 — Two-sided thresholding and RGB masking

This keeps intensity values between lower and upper thresholds.


In [ ]:
lower_threshold = 50
upper_threshold = 210
two_sided_mask = np.zeros_like(gray_8bit)
two_sided_mask[(gray_8bit > lower_threshold) & (gray_8bit < upper_threshold)] = 255
mask_rgb = cv2.cvtColor(two_sided_mask, cv2.COLOR_GRAY2RGB)
masked_rgb = (np.array(PIL_img) * (mask_rgb / 255.0)).astype(np.uint8)
img_grid([PIL_img, mask_rgb, masked_rgb],
         'Two-Sided Thresholding', ['Original', 'Mask', 'Original x Mask'], cols=3)


## Cell 21 — Inverse threshold masking

This applies the inverse threshold mask to keep the opposite image region.


In [ ]:
inverse_mask_rgb = skimage.util.invert(mask_rgb) / 255.0
inverse_output = (np.array(PIL_img) * inverse_mask_rgb).astype(np.uint8)
img_grid([PIL_img, inverse_mask_rgb, inverse_output],
         'Inverse Threshold Masking', ['Original', 'Inverse Mask', 'Output'], cols=3)


## Cell 22 — Histogram equalization

This improves global contrast and displays before/after histograms.


In [ ]:
def histogram_equalization(image):
    return (exposure.equalize_hist(image) * 255).astype(np.uint8)

equalized = histogram_equalization(gray_8bit)
img_grid([gray_8bit, equalized], 'Histogram Equalization', ['Original', 'Equalized'], cols=2)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); sns.histplot(gray_8bit.flatten(), bins=256, color='gray'); plt.title('Original Histogram')
plt.subplot(1, 2, 2); sns.histplot(equalized.flatten(), bins=256, color='gray'); plt.title('Equalized Histogram')
plt.tight_layout(); plt.show()


## Cell 23 — Histogram matching

This changes the source image histogram to resemble a reference image histogram.


In [ ]:
reference_url = 'https://media.geeksforgeeks.org/wp-content/uploads/20190721215512/sample.jpg'
reference = Image.open(BytesIO(requests.get(reference_url).content)).convert('L')
reference_np = np.array(reference)
matched = np.clip(match_histograms(gray_8bit, reference_np), 0, 255).astype(np.uint8)
img_grid([gray_8bit, reference_np, matched], 'Histogram Matching',
         ['Source', 'Reference', 'Matched'], cols=3)


# Lab 5 — Filtering, Noise, Denoising, Edges, and Sharpening


## Cell 24 — Mean filter / average blur

This applies a 9×9 mean filter using convolution.


In [ ]:
n = 9
mean_kernel = np.ones((n, n), dtype=np.float32) / (n * n)
mean_filtered = np.clip(convolve(gray_8bit, mean_kernel), 0, 255).astype(np.uint8)
img_grid([gray_8bit, mean_filtered], 'Mean Filtering', ['Original', 'Mean Filter 9x9'], cols=2)


## Cell 25 — Add salt-and-pepper and Gaussian noise

These helper functions generate common noisy images.


In [ ]:
def add_salt_and_pepper_noise(image, prob):
    noisy = random_noise(image / 255.0, mode='s&p', amount=prob)
    return (noisy * 255).astype(np.uint8)

def add_gaussian_noise(image, mean=0, var=0.1):
    noisy = random_noise(image / 255.0, mode='gaussian', mean=mean, var=var)
    return (noisy * 255).astype(np.uint8)

sp_low = add_salt_and_pepper_noise(gray_8bit, 0.02)
sp_medium = add_salt_and_pepper_noise(gray_8bit, 0.10)
sp_high = add_salt_and_pepper_noise(gray_8bit, 0.50)
gauss_low = add_gaussian_noise(gray_8bit, var=0.01)
gauss_medium = add_gaussian_noise(gray_8bit, var=0.05)
gauss_high = add_gaussian_noise(gray_8bit, var=0.50)

img_grid([gray_8bit, sp_low, sp_medium, sp_high], 'Salt-and-Pepper Noise', ['Original', 'Low', 'Medium', 'High'], cols=4)
img_grid([gray_8bit, gauss_low, gauss_medium, gauss_high], 'Gaussian Noise', ['Original', 'Low', 'Medium', 'High'], cols=4)


## Cell 26 — Median and Gaussian denoising

Use median filtering for salt-and-pepper noise and Gaussian blur for Gaussian noise.


In [ ]:
def apply_median_filter(image, size=3):
    return median_filter(image, size=size)

def apply_gaussian_filter(image, ksize=5, sigma=5):
    return cv2.GaussianBlur(image, (ksize, ksize), sigma)

sp_denoised = apply_median_filter(sp_low, 3)
gauss_denoised = apply_gaussian_filter(gauss_low, 5, 5)
img_grid([sp_low, sp_denoised, gauss_low, gauss_denoised],
         'Noise Removal', ['Salt & Pepper', 'Median Filtered', 'Gaussian Noise', 'Gaussian Filtered'], cols=2)


## Cell 27 — Prewitt edge detection

This detects horizontal and vertical edges with convolution kernels.


In [ ]:
prewitt_h = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]])
prewitt_v = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
edge_input = gray_8bit.astype(np.int16)
horizontal_edges = np.clip(convolve(edge_input, prewitt_h), 0, 255).astype(np.uint8)
vertical_edges = np.clip(convolve(edge_input, prewitt_v), 0, 255).astype(np.uint8)
img_grid([gray_8bit, horizontal_edges, vertical_edges],
         'Prewitt Edge Detection', ['Original', 'Horizontal Edges', 'Vertical Edges'], cols=3)


## Cell 28 — Laplacian filters and sharpening

This applies Laplacian kernels and then sharpens the image.


In [ ]:
laplace_kernel1 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]])
laplace_kernel2 = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]])
laplace_input = gray_8bit.astype(np.int16)
laplace_raw1 = convolve(laplace_input, laplace_kernel1)
laplace_raw2 = convolve(laplace_input, laplace_kernel2)
laplace1 = np.clip(laplace_raw1, 0, 255).astype(np.uint8)
laplace2 = np.clip(laplace_raw2, 0, 255).astype(np.uint8)
sharpen1 = np.clip(laplace_input - laplace_raw1, 0, 255).astype(np.uint8)
sharpen2 = np.clip(laplace_input + laplace_raw2, 0, 255).astype(np.uint8)
img_grid([gray_8bit, laplace1, laplace2, sharpen1, sharpen2],
         'Laplacian and Sharpening', ['Original', 'Laplacian 1', 'Laplacian 2', 'Sharpen 1', 'Sharpen 2'], cols=3)


## Cell 29 — Unsharp masking

Unsharp masking enhances image details.


In [ ]:
from skimage.filters import unsharp_mask
unsharp1 = unsharp_mask(gray_8bit, radius=1)
unsharp5 = unsharp_mask(gray_8bit, radius=5)
unsharp10 = unsharp_mask(gray_8bit, radius=10)
img_grid([gray_8bit, unsharp1, unsharp5, unsharp10],
         'Unsharp Masking', ['Original', 'Radius 1', 'Radius 5', 'Radius 10'], cols=4)


# Lab 6 — PSNR, DFT, DCT, Frequency Filtering, and Compression


## Cell 30 — PSNR calculation

A higher PSNR means the test image is closer to the original image.


In [ ]:
def PSNR(ground_truth, test_image):
    return skimage.metrics.peak_signal_noise_ratio(ground_truth, test_image)

print('PSNR original vs original:', PSNR(gray_8bit, gray_8bit))
print('PSNR original vs S&P high:', PSNR(gray_8bit, sp_high))
print('PSNR original vs Gaussian low:', PSNR(gray_8bit, gauss_low))


## Cell 31 — DFT / FFT and inverse FFT

This computes the Fourier transform, shifts it to the center, and reconstructs it.


In [ ]:
fft_input = gray_8bit
fft = np.fft.fft2(fft_input)
fft_shifted = np.fft.fftshift(fft)
fft_unshifted = np.fft.ifftshift(fft_shifted)
ifft = np.fft.ifft2(fft_unshifted)
img_grid([fft_input, np.log(np.abs(fft) + 1), np.log(np.abs(fft_shifted) + 1), np.abs(ifft)],
         'DFT / FFT', ['Original', 'FFT Magnitude', 'Shifted FFT', 'Reconstructed'], cols=4)


## Cell 32 — Create an ideal low-pass mask

This makes a circular mask centered on the Fourier spectrum.


In [ ]:
def ideal_low_pass_mask(shape, radius):
    height, width = shape
    cy, cx = height // 2, width // 2
    y, x = np.ogrid[:height, :width]
    return (((x - cx) ** 2 + (y - cy) ** 2) <= radius ** 2).astype(np.uint8)

radius = 50
ilpf_mask = ideal_low_pass_mask(fft_shifted.shape, radius)
img_grid([ilpf_mask], 'Ideal Low-Pass Mask', [f'Radius = {radius}'], cols=1, figsize=(6, 6))


## Cell 33 — Apply ideal low-pass filter (ILPF)

This keeps low-frequency information and produces a blurred output.


In [ ]:
fft_low = fft_shifted * ilpf_mask
ilpf_output = np.abs(np.fft.ifft2(np.fft.ifftshift(fft_low)))
img_grid([fft_input, ilpf_mask, np.log(np.abs(fft_low) + 1), ilpf_output],
         'Ideal Low-Pass Filter', ['Original', 'ILPF Mask', 'Filtered Spectrum', 'Output'], cols=4)
print('PSNR ILPF:', PSNR(fft_input, ilpf_output))


## Cell 34 — Apply ideal high-pass filter (IHPF)

This keeps high-frequency components and emphasizes edges/details.


In [ ]:
ihpf_mask = 1 - ilpf_mask
fft_high = fft_shifted * ihpf_mask
ihpf_output = np.abs(np.fft.ifft2(np.fft.ifftshift(fft_high)))
img_grid([fft_input, ihpf_mask, np.log(np.abs(fft_high) + 1), ihpf_output],
         'Ideal High-Pass Filter', ['Original', 'IHPF Mask', 'Filtered Spectrum', 'Output'], cols=4)


## Cell 35 — Compare ideal low-pass filters with different radii

Smaller radius gives more blur; larger radius keeps more details.


In [ ]:
def ideal_low_pass_filter(image, radius):
    f = np.fft.fft2(image)
    shifted = np.fft.fftshift(f)
    mask = ideal_low_pass_mask(shifted.shape, radius)
    return np.abs(np.fft.ifft2(np.fft.ifftshift(shifted * mask)))

lp10 = ideal_low_pass_filter(fft_input, 10)
lp20 = ideal_low_pass_filter(fft_input, 20)
lp50 = ideal_low_pass_filter(fft_input, 50)
img_grid([lp10, lp20, lp50], 'ILPF Radius Comparison', ['Radius 10', 'Radius 20', 'Radius 50'], cols=3)


## Cell 36 — Multiply the three low-pass outputs

This multiplies low-pass results and normalizes them for visualization.


In [ ]:
combined = lp10 * lp20 * lp50
combined = ((combined - combined.min()) / (combined.max() - combined.min()) * 255).astype(np.uint8)
img_grid([combined], 'Multiplied Low-Pass Results', ['R10 x R20 x R50'], cols=1, figsize=(8, 8))


## Cell 37 — Butterworth low-pass filter (BLPF)

Butterworth filtering uses a smooth frequency cutoff.


In [ ]:
def butterworth_low_pass_mask(shape, D0, n=2):
    height, width = shape
    cy, cx = height // 2, width // 2
    y, x = np.ogrid[:height, :width]
    distance = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    return 1 / (1 + (distance / D0) ** (2 * n))

D0 = 50
order = 2
blpf_mask = butterworth_low_pass_mask(fft_shifted.shape, D0, order)
blpf_output = np.abs(np.fft.ifft2(np.fft.ifftshift(fft_shifted * blpf_mask)))
img_grid([fft_input, blpf_mask, np.log(np.abs(fft_shifted * blpf_mask) + 1), blpf_output],
         'Butterworth Low-Pass Filter', ['Original', 'BLPF Mask', 'Spectrum', 'Output'], cols=4)


## Cell 38 — Butterworth high-pass filter (BHPF)

This inverts the Butterworth low-pass mask.


In [ ]:
bhpf_mask = 1 - blpf_mask
bhpf_output = np.abs(np.fft.ifft2(np.fft.ifftshift(fft_shifted * bhpf_mask)))
img_grid([fft_input, bhpf_mask, np.log(np.abs(fft_shifted * bhpf_mask) + 1), bhpf_output],
         'Butterworth High-Pass Filter', ['Original', 'BHPF Mask', 'Spectrum', 'Output'], cols=4)


## Cell 39 — Gaussian low-pass filter (GLPF)

Gaussian filtering has a very smooth cutoff.


In [ ]:
def gaussian_low_pass_mask(shape, D0):
    height, width = shape
    cy, cx = height // 2, width // 2
    y, x = np.ogrid[:height, :width]
    distance_squared = (x - cx) ** 2 + (y - cy) ** 2
    return np.exp(-distance_squared / (2 * D0 ** 2))

glpf_mask = gaussian_low_pass_mask(fft_shifted.shape, D0=50)
glpf_output = np.abs(np.fft.ifft2(np.fft.ifftshift(fft_shifted * glpf_mask)))
img_grid([fft_input, glpf_mask, np.log(np.abs(fft_shifted * glpf_mask) + 1), glpf_output],
         'Gaussian Low-Pass Filter', ['Original', 'GLPF Mask', 'Spectrum', 'Output'], cols=4)


## Cell 40 — Gaussian high-pass filter (GHPF)

This inverts the Gaussian low-pass mask to retain details.


In [ ]:
ghpf_mask = 1 - glpf_mask
ghpf_output = np.abs(np.fft.ifft2(np.fft.ifftshift(fft_shifted * ghpf_mask)))
img_grid([fft_input, ghpf_mask, np.log(np.abs(fft_shifted * ghpf_mask) + 1), ghpf_output],
         'Gaussian High-Pass Filter', ['Original', 'GHPF Mask', 'Spectrum', 'Output'], cols=4)


## Cell 41 — DCT and inverse DCT

This performs a 2D discrete cosine transform and reconstructs the image.


In [ ]:
def dct2(image):
    return dct(dct(image, axis=0, norm='ortho'), axis=1, norm='ortho')

def idct2(coefficients):
    return idct(idct(coefficients, axis=0, norm='ortho'), axis=1, norm='ortho')

dct_image = dct2(fft_input)
dct_output = np.abs(idct2(dct_image))
img_grid([fft_input, np.log(np.abs(dct_image) + 1), dct_output],
         'DCT Transform', ['Original', 'DCT Magnitude', 'Reconstructed'], cols=3)
print('PSNR DCT:', PSNR(fft_input, dct_output))


## Cell 42 — Frequency-domain compression

This retains only FFT coefficients above the 75th magnitude percentile.


In [ ]:
compression_fft = np.fft.fft2(fft_input)
compression_shifted = np.fft.fftshift(compression_fft)
compression_magnitude = np.abs(compression_shifted)
threshold = np.percentile(compression_magnitude, 75)
compression_mask = compression_magnitude >= threshold
compression_filtered = compression_shifted * compression_mask.astype(int)
compression_output = np.abs(np.fft.ifft2(np.fft.ifftshift(compression_filtered)))
compression_output_uint8 = np.clip(compression_output, 0, 255).astype(np.uint8)
img_grid([fft_input, np.log(compression_magnitude + 1), np.log(np.abs(compression_filtered) + 1), compression_output_uint8],
         'Frequency Compression', ['Original', 'FFT', 'Thresholded FFT', 'Output'], cols=4)
print('PSNR Compression:', PSNR(fft_input, compression_output_uint8))


# Lab 7 — Morphology, Segmentation, and Hough Transform


## Cell 43 — Erosion and dilation

Erosion shrinks bright regions and dilation expands bright regions.


In [ ]:
kernel_size = 15
morph_kernel = np.ones((kernel_size, kernel_size), dtype=np.uint8)
eroded = erosion(gray_8bit, footprint=morph_kernel)
dilated = dilation(gray_8bit, footprint=morph_kernel)
img_grid([gray_8bit, eroded, dilated], 'Morphological Operations', ['Original', 'Erosion', 'Dilation'], cols=3)


## Cell 44 — Global, Otsu, and adaptive thresholding

Global uses a fixed value; Otsu finds a global value; adaptive uses local values.


In [ ]:
from skimage.filters import threshold_otsu, threshold_local
page_image = data.page()
global_binary = page_image > 128
otsu_value = threshold_otsu(page_image)
otsu_binary = page_image > otsu_value
local_value = threshold_local(page_image, block_size=35, offset=10)
adaptive_binary = page_image > local_value
img_grid([page_image, global_binary * 255, otsu_binary * 255, adaptive_binary * 255],
         'Thresholding Methods', ['Original', 'Global 128', f'Otsu {otsu_value}', 'Adaptive'], cols=4)


## Cell 45 — Watershed segmentation

This uses a Sobel elevation map and markers to segment objects.


In [ ]:
coins_image = data.coins()
elevation = filters.sobel(coins_image)
markers = np.zeros_like(coins_image)
markers[coins_image < 30] = 1
markers[coins_image > 150] = 2
watershed_result = segmentation.watershed(elevation, markers)
img_grid([coins_image, elevation, markers * 127, watershed_result * 127],
         'Watershed Segmentation', ['Original', 'Sobel', 'Markers', 'Watershed'], cols=4)


## Cell 46 — Connected component labeling

This gives a separate integer label to every connected foreground region.


In [ ]:
coins_otsu = threshold_otsu(coins_image)
coins_binary = coins_image > coins_otsu
labeled_coins, count = label(coins_binary)
print('Connected components:', count)
img_grid([coins_binary * 255, labeled_coins], 'Connected Components', ['Binary', 'Labels'], cols=2)


## Cell 47 — Color overlay for labeled components

This visualizes connected-component labels using different colors.


In [ ]:
overlay = label2rgb(labeled_coins, image=coins_image, bg_label=0)
overlay_uint8 = (overlay * 255).astype(np.uint8)
img_grid([overlay_uint8], 'Label Overlay', ['Colored Components'], cols=1, figsize=(9, 8))


## Cell 48 — Hough circle transform

This finds circles from Canny edges and draws detected circles in red.


In [ ]:
hough_image = data.coins()
edges = canny(hough_image, sigma=2, low_threshold=10, high_threshold=50)
hough_radii = np.arange(10, 60, 2)
hough_result = hough_circle(edges, hough_radii)
accums, cx, cy, radii = hough_circle_peaks(hough_result, hough_radii, total_num_peaks=30)

circle_output = color.gray2rgb(hough_image)
for center_y, center_x, radius in zip(cy, cx, radii):
    circle_y, circle_x = circle_perimeter(center_y, center_x, radius, shape=circle_output.shape)
    circle_output[circle_y, circle_x] = (220, 20, 20)

img_grid([hough_image, edges * 255, circle_output],
         'Hough Circle Transform', ['Original', 'Canny Edges', 'Detected Circles'], cols=3)


## Cell 49 — Optional Hough line transform

Run this if the exam asks for straight-line detection.


In [ ]:
from skimage.transform import probabilistic_hough_line
line_image = data.camera()
line_edges = canny(line_image, sigma=2)
lines = probabilistic_hough_line(line_edges, threshold=10, line_length=5, line_gap=3)
plt.figure(figsize=(8, 8))
plt.imshow(line_image, cmap='gray')
for p0, p1 in lines:
    plt.plot((p0[0], p1[0]), (p0[1], p1[1]), 'r-')
plt.axis('off')
plt.title('Hough Line Detection')
plt.show()


# Quick Exam Index

- Cells 1–10: imports, reading images, grayscale, channels, resize, intensity, histogram
- Cells 11–18: operations, bit planes, masks, log/gamma, stretching
- Cells 19–23: thresholding, equalization, matching
- Cells 24–29: filtering, noise, denoising, edges, sharpening
- Cells 30–42: PSNR, FFT/DFT, DCT, compression, ideal/Butterworth/Gaussian filters
- Cells 43–49: morphology, adaptive thresholding, watershed, labels, Hough transforms
